# 📝 그래프 집계 과제 LV3(통합): 추천 엔진·판매 리포트

> 이 단원의 집계·랭킹·인덱스를 엮어 **작은 프로그램 두 개**를 완성합니다. 함께 구매를 기반으로 한 **추천 엔진**과, 여러 집계를 모은 **판매 대시보드 리포트**.

## 풀이 방법
1. 맨 위 **준비 셀 3개**(연결 → 초기화 → 시드 적재)를 먼저 실행하세요. 반드시 **실습 전용 DB**로.
2. 각 문제는 **단계별 셀**로 나뉩니다. 각 단계 셀의 지시대로 채우세요.
3. 산출물은 `output/` 폴더에 저장됩니다.

- 도메인: **이커머스**: 구매자(Buyer)가 상품(Product, price·category)을 구매(PURCHASED{qty})합니다.

화이팅!

아래 준비 셀이 만들 그래프의 전체 모습입니다. 구매자 5명과 상품 6개가 구매 관계 12건으로 이어져 있습니다. **한 상품을 함께 산 사람들**을 세는 것이 1번 추천 엔진의 뼈대입니다.

<img src="images/그래프_한눈에_커머스.png" width="820">

아래 준비 셀 3개를 위에서부터 실행하세요.

In [ ]:
# [제공 코드] Neo4j 연결: 실행만 하세요. .env 로 연결하고 run_cypher 헬퍼를 만듭니다.
# 반드시 "실습 전용" DB 에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase
from neo4j.exceptions import ConstraintError  # UNIQUE 제약 위반 에러

# 1) 접속 정보: .env 를 환경변수로 올린다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# .env 를 못 읽어도 에러가 아니라 기본값으로 넘어간다. 마지막 줄의 주소를 눈으로 확인할 것
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북이 끝날 때까지 재사용할 통로 하나
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 키는 RETURN 의 별칭이다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 그래프 초기화: 실습 전용 DB 인지 꼭 확인하고 실행하세요!
# 노드·관계에 더해 이 노트북이 만든 인덱스·제약까지 지웁니다(DB 기본 LOOKUP 인덱스는 그대로).
# 1) 제약 먼저. 제약이 남아 있으면 그 제약이 만든 인덱스를 따로 못 지운다
for _row in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name"):
    run_cypher("DROP CONSTRAINT " + _row["name"] + " IF EXISTS")
# 2) 남은 인덱스. LOOKUP 은 DB 기본이라 뺀다
for _row in run_cypher("SHOW INDEXES YIELD name, type WHERE type <> 'LOOKUP' RETURN name"):
    run_cypher("DROP INDEX " + _row["name"] + " IF EXISTS")
# 3) 노드·관계. 관계가 9만 개라 한 번에 담지 않고 2만 개씩 끊어 지운다
while True:
    # DETACH DELETE 는 매달린 관계까지 함께 지운다
    _left = run_cypher("MATCH (n) WITH n LIMIT 20000 DETACH DELETE n RETURN count(n) AS n")[0]["n"]
    if _left == 0:
        break
print("초기화 완료: 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

In [ ]:
# [제공 코드] 이커머스 시드 적재: 실행만 하세요.
# Buyer -PURCHASED{qty}-> Product{code, price, category}
products = [
    ["무선이어폰", "P01", 80000, "가전"],
    ["블루투스스피커", "P02", 60000, "가전"],
    ["노트북거치대", "P03", 40000, "가전"],
    ["텀블러", "P04", 20000, "리빙"],
    ["담요", "P05", 30000, "리빙"],
    ["머그컵", "P06", 15000, "리빙"],
]
buyers = ["도윤", "하윤", "지호", "서아", "민재"]
purchased = [
    ["도윤", "무선이어폰", 1], ["도윤", "텀블러", 2],
    ["하윤", "무선이어폰", 2], ["하윤", "텀블러", 1], ["하윤", "블루투스스피커", 1],
    ["지호", "무선이어폰", 1], ["지호", "담요", 2],
    ["서아", "머그컵", 3], ["서아", "담요", 1], ["서아", "노트북거치대", 1],
    ["민재", "무선이어폰", 1], ["민재", "텀블러", 1],
]
for name, code, price, category in products:
    run_cypher("MERGE (pr:Product {name: $name}) SET pr.code = $code, pr.price = $price, pr.category = $category",
               name=name, code=code, price=price, category=category)
for name in buyers:
    run_cypher("MERGE (:Buyer {name: $name})", name=name)
for buyer, product, qty in purchased:
    run_cypher("MATCH (b:Buyer {name: $buyer}), (pr:Product {name: $product}) "
               "MERGE (b)-[x:PURCHASED]->(pr) SET x.qty = $qty", buyer=buyer, product=product, qty=qty)
print("적재 완료: 상품:", run_cypher("MATCH (pr:Product) RETURN count(pr) AS n")[0]["n"],
      "· 구매:", run_cypher("MATCH ()-[x:PURCHASED]->() RETURN count(x) AS n")[0]["n"])


## 그래프 살펴보기

문제로 들어가기 전에 어떤 레이블·관계·속성이 있는지 한 번 훑습니다. 어디에 인덱스를 걸지, 무엇을 그룹핑 키로 둘지가 여기서 정해집니다.

In [ ]:
# [제공 코드] 그래프에 무엇이 들어 있는지 훑어봅니다(실행만 하세요).
# 1) 레이블마다 몇 개인지. 노드마다 레이블이 하나뿐이라 labels(n)[0] 로 충분하다
for row in run_cypher("MATCH (n) RETURN labels(n)[0] AS 레이블, count(*) AS 개수 ORDER BY 레이블"):
    print(row)

# 2) 관계 타입마다 몇 개인지. type(x) 가 관계 종류 이름이다
for row in run_cypher("MATCH ()-[x]->() RETURN type(x) AS 관계, count(x) AS 개수 ORDER BY 관계"):
    print(row)

# 3) 관계에 붙은 속성 이름. keys(x) 가 그 목록이다
for row in run_cypher("MATCH ()-[x]->() "
                      "RETURN type(x) AS 관계, collect(DISTINCT keys(x)) AS 속성키 ORDER BY 관계"):
    print(row)


아래 셀은 실행계획을 보는 헬퍼입니다(교안_02 에서 쓴 것과 같습니다). 1단계에서 씁니다.

In [ ]:
# [제공 코드] 실행계획 보기 헬퍼: 실행만 하세요.
# explain_plan(쿼리, profile=False) 는 계획만, True 는 비용(dbHits)까지. quiet=True 면 찍지 않고 값만.
PLAN_CALLS = []   # 부른 기록: (쿼리, profile 여부, 연산자 목록).
#                 과제 채점이 "정말 재 봤는지"와 "그 결과를 담았는지"를 볼 때 쓴다


def explain_plan(query, profile=True, quiet=False, **params):
    """실행계획을 출력하고 (연산자 이름 리스트, 총 dbHits) 를 돌려준다."""
    # profile=True 는 PROFILE(실제로 실행하며 dbHits 측정), False 는 EXPLAIN(실행 없이 계획만)
    with driver.session() as session:
        if profile:
            result = session.run("PROFILE " + query, **params)
            list(result)                      # PROFILE 은 결과를 끝까지 읽어야 계측이 끝난다
            plan = result.consume().profile   # 계획은 실행이 끝난 뒤에야 받을 수 있다
        else:
            plan = session.run("EXPLAIN " + query, **params).consume().plan
    total = 0        # 계획 전체의 dbHits 합
    operators = []   # 위에서부터 만난 연산자 이름들. 뒤에서 'NodeIndexSeek 이 있나' 를 볼 때 쓴다

    # 실행계획은 나무 모양이라, 자기 자신을 다시 부르며 아래로 내려간다
    def walk(node, depth=0):
        nonlocal total
        hits = node.get("dbHits")                   # EXPLAIN 으로 뽑은 계획에는 이 값이 없다(None)
        total += hits or 0
        name = node["operatorType"].split("@")[0]   # 'NodeIndexSeek@neo4j' 에서 이름만 남긴다
        operators.append(name)
        cost = f"| dbHits = {hits}" if profile else ""
        if not quiet:                     # quiet=True 면 재기만 하고 찍지 않는다
            print("  " * depth, name, cost)
        for child in node.get("children", []):
            walk(child, depth + 1)        # 자식 계획은 한 칸 더 들여써 찍는다

    walk(plan)
    # 무엇을 쟀고 무엇이 나왔는지 남긴다(채점이 손으로 적은 목록을 걸러 낼 때 본다)
    PLAN_CALLS.append((query, profile, operators))
    if profile and not quiet:
        print("총 dbHits:", total)
    return operators, total

In [ ]:
# [제공 코드] 1단계에서 실행계획을 대조할 대상 조회입니다(실행만 하세요).
# 상품을 '이름으로 찾는' 조회라, Product.name 인덱스가 있으면 훑기 대신 색인으로 바로 갑니다.
TARGET_QUERY = "MATCH (pr:Product {name: '무선이어폰'}) RETURN pr.name AS 상품"
print(run_cypher(TARGET_QUERY))

---
## 1. 함께 구매 추천 엔진
**배경**: "이 상품을 산 사람들이 **함께 산 상품**"을 추천하는 작은 엔진을 만듭니다. 인덱스로 조회를 빠르게 하고, 상품명을 받아 추천 목록을 돌려주는 함수를 정의합니다. 아래 **단계별 셀**을 순서대로 채우세요.

### 1단계: 상품명 인덱스 설계 (멱등) + 실행계획으로 확인
상품을 이름으로 자주 찾으니 `Product.name` 에 인덱스 **`product_name_idx`** 를 만드세요(`IF NOT EXISTS`: 여러 번 실행해도 안전한 **멱등** 생성). 그런데 "만들었다"로 끝내지 말고, **정말 그 인덱스를 타는지 실행계획으로 확인**합니다.

- 먼저 `TARGET_QUERY` 로 준비된 조회의 실행계획을 `explain_plan(TARGET_QUERY, profile=False)` 로 찍어, 돌려받은 연산자 이름 리스트를 **`plan_before`** 에 담으세요.
- `Product.name` 에 인덱스 **`product_name_idx`** 를 만들고(`IF NOT EXISTS`) `db.awaitIndexes()` 로 기다린 뒤, `SHOW INDEXES` 로 인덱스 이름 목록을 **`index_names`**(문자열 리스트)에 담으세요(LOOKUP 제외).
- **같은 쿼리**로 실행계획을 다시 찍어 **`plan_after`** 에 담으세요.

**예시**: `plan_before` 에는 `NodeByLabelScan`(상품을 전부 훑음)이, `plan_after` 에는 `NodeIndexSeek`(색인으로 바로 도달)가 들어 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안의 인덱스 3단계: CREATE INDEX(IF NOT EXISTS) → db.awaitIndexes → SHOW INDEXES.
- 그 앞뒤로 실행계획을 한 번씩 찍어 연산자 이름이 어떻게 바뀌는지 남긴다.

세부구현:
1. 준비 셀이 만들어 둔 TARGET_QUERY 를 헬퍼에 넘겨 연산자 목록을 받아 plan_before 에 담는다.
   1-1. 헬퍼는 (연산자 이름 리스트, 총 dbHits) 두 개를 돌려준다.
   1-2. 계획만 볼 것이므로 profile 인자를 False 로 준다.
2. CREATE INDEX 로 Product 의 name 속성에 인덱스(이름 product_name_idx, IF NOT EXISTS)를 만든다.
3. CALL db.awaitIndexes() 로 기다린다.
4. SHOW INDEXES 로 LOOKUP 을 제외한 이름을 조회해 index_names(문자열 리스트)로 만든다.
5. 같은 TARGET_QUERY 로 1번을 한 번 더 해서 plan_after 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

### 2단계: 상품 코드 UNIQUE 제약 (설계 근거)
상품에는 이름과 별개로 **상품 코드**(`code`, 예: `P01`)가 있습니다. 이 시스템은 **`name` 에는 인덱스**를, **`code` 에는 UNIQUE 제약**을 겁니다.

- `Product.code` 에 UNIQUE 제약 **`product_code_unique`** 를 만드세요(`IF NOT EXISTS`). `db.awaitIndexes()` 로 기다린 뒤, `SHOW CONSTRAINTS` 로 제약 이름 목록을 **`constraint_names`**(문자열 리스트)에 담으세요.
- 그런 다음 **아래 서술 셀**에 답하세요(자동 채점 없음, 정답 노트북의 모범 서술과 비교):
  **왜 `name` 에는 인덱스를 걸고 `code` 에는 제약을 걸었을까요?** 두 기능이 각각 무엇을 보장하는지,   그리고 두 속성의 성격이 어떻게 다른지를 근거로 두세 문장으로 쓰세요.

<details><summary>힌트</summary>

```text
접근방법:
- 교안의 UNIQUE 제약: CREATE CONSTRAINT ... REQUIRE ... IS UNIQUE → db.awaitIndexes → SHOW CONSTRAINTS.

세부구현:
1. CREATE CONSTRAINT 로 Product 의 code 가 UNIQUE 하도록 제약(이름 product_code_unique, IF NOT EXISTS)을 만든다.
2. CALL db.awaitIndexes() 로 기다린다.
3. SHOW CONSTRAINTS 로 제약 이름을 조회해 constraint_names(문자열 리스트)로 만든다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

*(여기에 자신의 답을 서술하세요. 정답 노트북의 모범 서술과 비교해 보세요.)*

### 3단계: 추천 함수 정의
상품명을 받아 **함께 구매한 상품 랭킹**을 돌려주는 함수 **`recommend(product_name)`** 를 정의하세요.
- 패턴: `(대상상품)<-[:PURCHASED]-(구매자)-[:PURCHASED]->(다른상품)` 에서 대상상품 자신은 제외.
- 다른 상품마다 **함께 산 구매자 수**(`count(DISTINCT b)`)를 세어, **내림차순**(같으면 상품명 오름차순)으로 정렬한 결과(dict 리스트)를 돌려줍니다. 별칭은 **`추천상품`**·**`함께구매수`**.
- 상품명은 함수의 매개변수로 받아 **파라미터로 넘깁니다**. 쿼리 문자열에 이어 붙이지 마세요. 쿼리 안에는 `$name` 같은 자리표시자를 두고, 값은 `run_cypher(쿼리, name=product_name)` 으로 따로 넘깁니다(교안_02 3절과 시드 적재 셀이 쓰는 방식).

<details><summary>힌트</summary>

```text
접근방법:
- 교안의 공유 패턴 랭킹(가운데 노드를 세어 줄 세우기)을 상품으로 옮긴다(대상<-구매-구매자-구매->다른상품). 대상 이름은 쿼리에 이어 붙이지 말고 파라미터로 넘긴다.

세부구현:
1. def recommend(product_name): 안에서 run_cypher 결과를 그대로 return 한다.
2. MATCH 로 '대상 상품 ← 구매자 → 다른 상품' 공동구매 패턴을 잡고, WHERE 로 다른 상품이 대상 자신이 아니게 한다.
3. RETURN 에 다른 상품 이름과 서로 다른 구매자 수(count DISTINCT)를 별칭(추천상품·함께구매수)으로 두고, 함께구매수 내림차순(같으면 추천상품순)으로 ORDER BY 한다.
4. 대상 이름은 쿼리 안에 자리표시자로 두고, run_cypher 의 이름 붙인 인자로 product_name 을 넘긴다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

### 4단계: 추천 실행·검증
**`무선이어폰`** 을 산 사람들에게 추천할 상품을 `recommend('무선이어폰')` 로 구해 **`rec`** 에 담으세요. 가장 함께 많이 산 상품이 1위로 나와야 합니다.
- 이어서 **다른 상품명으로도 한 번 더** 호출해 `recommend('담요')` 의 결과를 **`rec2`** 에 담으세요. 상품명이 정말 **매개변수로 흐르는지**는 두 번째 호출에서만 드러납니다.

**예시**: `rec[0]` 은 추천상품 **텀블러**, 함께구매수 **3** 입니다. `rec2` 는 **3행**이고 함께구매수가 모두 같아, 순서는 보조 정렬 키가 정합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 3단계에서 만든 recommend 를 서로 다른 상품명으로 두 번 호출해 rec, rec2 에 담는다.

세부구현:
1. rec = recommend('무선이어폰')
2. rec2 = recommend('담요')
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert 'product_name_idx' in index_names, '인덱스 이름을 product_name_idx 로 만들었는지 확인하세요'
assert 'product_code_unique' in constraint_names, '제약 이름을 product_code_unique 로 만들었는지 확인하세요'
# 실행계획이 실제로 바뀌었는지 본다. '만들었다'가 아니라 '탄다'를 확인하는 자리다
planned_calls = [q for q, profiled, ops in PLAN_CALLS if q == TARGET_QUERY]
measured = [ops for _, _, ops in PLAN_CALLS]
assert len(planned_calls) >= 2, 'TARGET_QUERY 의 실행계획을 explain_plan 으로 인덱스 만들기 전·후 두 번 재야 합니다(plan_before·plan_after 를 손으로 적어 넣으면 여기서 걸립니다)'
assert plan_before in measured and plan_after in measured, '두 변수에는 explain_plan 이 돌려준 목록을 그대로 담으세요(손으로 적은 목록은 통과하지 못합니다)'
assert 'NodeByLabelScan' in plan_before, '인덱스를 만들기 전 계획에는 NodeByLabelScan 이 있어야 합니다. plan_before 를 먼저 찍었는지, 답안 셀을 두 번째로 실행한 것은 아닌지 확인하세요(인덱스가 이미 있으면 맨 위 준비 셀부터 다시 실행해야 합니다)'
assert 'NodeIndexSeek' in plan_after, '인덱스를 만든 뒤 계획에 NodeIndexSeek 가 없습니다. db.awaitIndexes() 로 기다린 뒤 다시 찍었는지 확인하세요'
# 파이썬 변수뿐 아니라 DB 를 다시 조회해 인덱스·제약이 실제로 살아 있는지 확인한다
live_idx = run_cypher("SHOW INDEXES YIELD name, state RETURN name, state")
hit = [r for r in live_idx if r['name'] == 'product_name_idx']
assert len(hit) == 1 and hit[0]['state'] == 'ONLINE', '인덱스 상태가 ONLINE 이 아닙니다. CALL db.awaitIndexes() 를 불렀는지 확인하세요'
live_con = run_cypher("SHOW CONSTRAINTS YIELD name, properties "
                      "RETURN name, properties")
con = [r for r in live_con if r['name'] == 'product_code_unique']
assert len(con) == 1 and con[0]['properties'] == ['code'], '제약이 code 속성에 걸렸는지 확인하세요'
assert rec[0]['추천상품'] == '텀블러' and rec[0]['함께구매수'] == 3, '1위 추천이 다릅니다. 대상 상품 자신을 제외했는지, 사람 수를 DISTINCT 로 셌는지 확인하세요'
assert rec == sorted(rec, key=lambda r: (-r['함께구매수'], r['추천상품'])), '정렬을 확인하세요: 함께구매수 내림차순, 같으면 상품명 오름차순'
# 두 번째 호출: 상품명이 쿼리에 박혀 있으면 여기서 첫 호출과 같은 결과가 나와 걸린다
assert [(r['추천상품'], r['함께구매수']) for r in rec2] == [('노트북거치대', 1), ('머그컵', 1), ('무선이어폰', 1)], '담요 추천 결과가 다릅니다. 상품명을 파라미터로 넘겼는지, 보조 정렬 키를 두었는지 확인하세요'
print('✅ 통과!')

### 5단계: 무엇으로 줄 세울 것인가 (비율 점수와 최소 조건)
지금 추천은 **함께 산 사람이 몇 명인가**(원점수)로 줄 세웁니다. 이 점수에는 알고 써야 할 성질이 있습니다. **원래 많이 팔린 상품이 유리**합니다. 다른 점수를 하나 더 만들어 견줍니다.

- (1) `recommend_ratio(product_name, min_buyers=2)` 함수를 정의하세요. 3단계의 공유 패턴으로 `함께구매수` 를 낸 뒤, `MATCH (other)<-[:PURCHASED]-(x:Buyer)` 를 이어 그 상품의 **전체 구매자 수**를 세고, `함께구매수 / 전체구매자수` 를 냅니다.
  - 별칭은 **`추천상품`**·**`함께구매수`**·**`전체구매자수`**·**`겹침비율`**. `겹침비율` 은 **소수 셋째 자리까지 `round`**.
  - ⚠️ **정수끼리 나누면 소수점 아래가 버려집니다.** `round(toFloat(함께구매수) / 전체구매자수, 3)` 처럼 한쪽을 `toFloat` 로 감싸세요.
  - 정렬은 **겹침비율 내림차순, 같으면 함께구매수 내림차순, 그다음 상품명 오름차순**.
  - `min_buyers` 는 **최소 조건**입니다. `전체구매자수` 가 그 값보다 작은 후보는 **빼세요**(집계한 뒤의 조건이니 `WITH` 뒤 `WHERE` 자리). 값은 파라미터로 넘깁니다.
- (2) `recommend_ratio('무선이어폰', min_buyers=1)` 을 **`ratio_all`** 에, `recommend_ratio('무선이어폰', min_buyers=2)` 를 **`ratio_guarded`** 에 담으세요.
- (3) 아래 서술 셀에 답하세요(자동 채점 없음): **원점수와 비율 중 무엇을 쓰겠습니까?** 두 점수가 각각 어느 쪽으로 치우치는지와, 최소 조건을 왜 두어야 하는지를 `ratio_all` 의 결과를 근거로 두세 문장으로 쓰세요.

**예시**: `ratio_all` 은 **3행**이고, 그중 `블루투스스피커` 은 전체 구매자가 **1명뿐**인데 겹침비율이 **1.0** 로 맨 위까지 올라옵니다. `ratio_guarded` 는 그 후보가 빠져 **2행**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 3단계 함수를 복사해 뒤에 두 단계를 더 붙인다: 후보의 전체 구매자 수를 세고, 나눈다.
- 최소 조건은 집계한 뒤에야 알 수 있는 값이 조건이므로 WITH 뒤의 WHERE 에 쓴다.

세부구현:
1. def recommend_ratio(product_name, min_buyers=2): 로 정의한다(한 줄 docstring).
2. 3단계와 같은 공유 패턴을 잡고 WITH other, count(DISTINCT b) AS 함께구매수 로 1차 집계.
3. MATCH (other)<-[:PURCHASED]-(x:Buyer) 를 잇고
   WITH other, 함께구매수, count(DISTINCT x) AS 전체구매자수 로 2차 집계.
4. WHERE 전체구매자수 >= $min_buyers 로 너무 작은 후보를 뺀다.
5. RETURN 에 round(toFloat(함께구매수) / 전체구매자수, 3) 을 겹침비율로 두고 정렬한다.
6. 값은 $name·$min_buyers 파라미터로 넘긴다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert [(r['추천상품'], r['함께구매수'], r['전체구매자수'], r['겹침비율']) for r in ratio_all] == [('텀블러', 3, 3, 1.0), ('블루투스스피커', 1, 1, 1.0), ('담요', 1, 2, 0.5)], '최소 조건 없이 낸 비율 랭킹이 다릅니다. 전체 구매자 수를 1차 집계 뒤에 이어 붙였는지, toFloat 로 감싸 나눴는지, 정렬 기준을 그대로 따랐는지 확인하세요'
assert [(r['추천상품'], r['함께구매수'], r['전체구매자수'], r['겹침비율']) for r in ratio_guarded] == [('텀블러', 3, 3, 1.0), ('담요', 1, 2, 0.5)], '최소 조건을 건 결과가 다릅니다. WHERE 전체구매자수 >= $min_buyers 를 WITH 뒤에 두었는지 확인하세요'
# 정수 나눗셈을 그대로 쓴 답안은 여기서 걸린다(비율이 전부 0 이나 1 이 된다)
assert any(0 < r['겹침비율'] < 1 for r in ratio_all), '겹침비율이 전부 0 또는 1 입니다. 정수끼리 나눠 소수점이 버려졌습니다. toFloat 를 쓰세요'
assert len(ratio_guarded) < len(ratio_all), 'min_buyers 를 올렸는데 후보가 줄지 않았습니다. 조건이 실제로 걸렸는지 확인하세요'
# 매개변수가 정말 흐르는지: 다른 상품으로 불러 결과가 달라져야 한다
assert recommend_ratio('담요', min_buyers=1) != ratio_all, '상품명이 쿼리에 박혀 있습니다. $name 파라미터로 넘겼는지 확인하세요'
print('✅ 통과!')

*(여기에 자신의 답을 서술하세요. 정답 노트북의 모범 서술과 비교해 보세요.)*

---
## 2. 판매 대시보드 리포트
**배경**: 매출을 여러 각도로 집계해 하나의 **리포트**로 모으고 파일로 저장합니다. 매출은 `구매수량(qty) × 상품가격(price)` 으로 계산합니다. 아래 단계를 순서대로 채우세요.

### 1단계: 매출 상위 3 상품 (안정 정렬)
상품별 매출 `sum(x.qty * pr.price)` 를 구해 **매출 내림차순**(같으면 상품명 오름차순)으로 정렬하고, 상위 3개 상품명을 리스트 **`top3`** 에 담으세요(문자열 리스트).

**예시**: `top3` 는 **['무선이어폰', '담요', '텀블러']** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 상품 이름을 그룹핑 키로 두고 매출(qty×price)을 sum 한 뒤 정렬·LIMIT 로 상위 3만 남기고 이름만 뽑는다.

세부구현:
1. MATCH 로 구매자→상품(PURCHASED) 을 잡고 구매 관계에 변수를 붙인다.
2. RETURN 에 상품 이름과 매출 합(sum 안에서 qty와 price 를 곱함)을 별칭(상품·매출)으로 둔다.
3. 매출 내림차순(같으면 상품순)으로 ORDER BY 하고 LIMIT 로 3개만 남긴 뒤, 상품명만 리스트로 뽑아 top3 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert top3 == ['무선이어폰', '담요', '텀블러'], '상위 3 상품이 다릅니다. 매출은 sum(qty * price) 이고, 정렬은 매출 내림차순(같으면 상품명순)입니다'
print('✅ 통과!')

### 2단계: 카테고리별 매출
카테고리별 매출 합계를 구해 사전 **`cat_revenue`**(키=카테고리, 값=매출)에 담으세요.

**예시**: `cat_revenue['가전']` 은 **500000**, `cat_revenue['리빙']` 은 **215000** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 카테고리를 그룹핑 키로 두고 매출(qty×price)을 sum 한 뒤, 그 결과를 파이썬 사전으로 만든다.

세부구현:
1. MATCH 로 구매자→상품(PURCHASED) 을 잡고 구매 관계에 변수를 붙인다.
2. RETURN 에 상품의 category 와 매출 합(sum)을 별칭(카테고리·매출)으로 둔다.
3. run_cypher 결과를 순회해 { 카테고리: 매출 } 사전으로 만들어 cat_revenue 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert cat_revenue['가전'] == 500000 and cat_revenue['리빙'] == 215000, '카테고리별 매출이 다릅니다. 상품의 category 를 그룹핑 키로 두고 sum(qty * price) 를 냈는지 확인하세요'
print('✅ 통과!')

### 3단계: 리포트 저장
위 결과와 **총매출**을 한 사전 **`report`** 로 모아 `output/ecommerce_report.json` 에 저장하세요. 키는 다음 셋입니다.
- `"best_sellers"`: 매출 상위 3 상품 리스트(`top3`)
- `"category_revenue"`: 카테고리별 매출 사전(`cat_revenue`)
- `"total_revenue"`: 전체 매출 합계(모든 구매의 `qty * price` 합)

**예시**: `report['total_revenue']` 는 **715000** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 총매출을 한 번 더 집계하고, 세 값을 사전으로 묶어 json 으로 저장한다.

세부구현:
1. 구매자→상품(PURCHASED) 전체에 대해 매출 합(sum(qty×price))을 한 번 더 집계해 total 로 꺼낸다.
2. report = {'best_sellers': top3, 'category_revenue': cat_revenue, 'total_revenue': total} 로 묶는다.
3. output 폴더를 만들고(os.makedirs) output/ecommerce_report.json 에 json.dumps 로 저장(UTF-8).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import json
from pathlib import Path
saved = json.loads(Path('output/ecommerce_report.json').read_text(encoding='utf-8'))
# 파일만 보면 지난 실행이 남긴 결과로도 통과한다. 이번에 만든 report 와 앞 단계 값까지 함께 대조한다
assert saved == report, '저장한 파일 내용이 report 와 다릅니다. report 를 그대로 json 으로 저장했는지 확인하세요'
assert report['best_sellers'] == top3, "report['best_sellers'] 에 1단계의 top3 를 그대로 넣었는지 확인하세요"
assert report['category_revenue'] == cat_revenue, "report['category_revenue'] 에 2단계의 cat_revenue 를 그대로 넣었는지 확인하세요"
assert saved['best_sellers'] == ['무선이어폰', '담요', '텀블러'], '저장된 best_sellers 가 top3 와 다릅니다'
assert saved['total_revenue'] == 715000, '총매출이 다릅니다. 모든 구매의 qty * price 를 더했는지 확인하세요'
assert saved['category_revenue']['가전'] == 500000, '저장된 category_revenue 가 cat_revenue 와 다릅니다'
print('✅ 통과!')

### 4단계: 리스트로 접어 검산하기
리포트를 냈으면 끝이 아니라 **맞는지 다시 재 봅니다.** 지금까지는 `sum` 으로 **여러 행**을 접었습니다. 이번에는 같은 값을 **한 행 안의 리스트**로 접어 다른 길로 재고, 3단계의 `total_revenue` 와 같은 수가 나오는지 봅니다.

**쓸 함수** (교안_01 7-2 리스트 갈래에서 본 것들입니다)

| 함수 | 하는 일 |
|---|---|
| `collect(x)` | 여러 행을 리스트 한 칸으로 접는다(이미 여러 번 썼습니다) |
| `head(목록)` · `last(목록)` | 리스트의 **첫 값**·**마지막 값**을 꺼낸다 |
| `reduce(...)` | 리스트를 **하나의 값으로** 접는다 |
| `range(시작, 끝)` | 숫자 리스트를 만든다. **끝을 포함**한다(`range(0, 2)` 는 `[0, 1, 2]`) |

`reduce` 의 모양은 `reduce(합 = 0, v IN 목록 | 합 + v)` 입니다. `합` 은 값을 쌓아 둘 이름이고 `0` 이 그 시작값입니다. **`sum` 은 여러 행을 접고, `reduce` 는 한 행 안의 리스트를 접습니다.**

**요구사항**:
- **(1)** 상품별 매출을 **매출 내림차순**(같으면 상품명 오름차순)으로 줄 세운 뒤, 상품 이름과 매출을 각각 `collect` 로 접어 **한 행**으로 만드세요. 그 행에서 `head`·`last` 로 매출 **최고 상품**과 **최저 상품**을, `reduce` 로 매출 **합계**를 내어 **첫 행(dict)** 을 **`check`** 에 담으세요. 별칭은 **`최고상품`**·**`최저상품`**·**`합계`**.
  - 접기 **전에** 줄을 세워야 `head` 가 1위가 됩니다(`collect` 는 담기는 순서를 보장하지 않습니다).
- **(2)** 같은 상품 목록에 `range` 로 **자리 번호**를 만들어 **상위 3위 순위표**를 **`ranked`** 에 담으세요. 별칭은 **`순위`**·**`상품`** 이고 순위는 **1부터** 시작합니다. (리스트의 자리 번호는 0 부터라, 순위로 쓰려면 1 을 더합니다.)

**예시**: (1) `check['합계']` 는 **715000** 으로 3단계의 `total_revenue` 와 같습니다. `최고상품` 은 **무선이어폰**, `최저상품` 은 **노트북거치대** 입니다. (2) `ranked` 의 상품만 뽑으면 **['무선이어폰', '담요', '텀블러']** 으로 **1단계의 `top3` 와 같은 답**입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 1단계와 같은 집계로 시작한다. 다른 점은 RETURN 대신 WITH 로 넘겨 collect 로 한 번 더 접는 것.
- ORDER BY 는 collect 보다 위에 있어야 한다(LV2 13번 그룹별 상위 N 에서 본 그 순서).
- (2) 는 만들어 둔 상품목록을 그대로 쓰면 된다. range 로 자리 번호를 펼쳐 목록[i] 로 꺼낸다.

세부구현:
1. MATCH 로 구매자→상품(PURCHASED) 을 잡고 WITH 로 상품 이름과 매출 합을 넘긴다.
2. 이어서 ORDER BY 로 매출 내림차순(같으면 상품명) 정렬한다.
3. WITH 로 상품 이름 리스트와 매출 리스트를 각각 collect 한다.
4. RETURN 에 head·last·reduce 를 요구사항의 별칭으로 두고 [0] 으로 첫 행을 꺼내 check 에 담는다.
5. (2) 는 3번까지 같게 만든 뒤 UNWIND range(0, 2) AS i 로 자리 번호를 펼치고,
   RETURN 에 i + 1 과 상품목록[i] 를 별칭 순위·상품 으로 둔다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert check['합계'] == 715000, 'reduce 로 접은 합계가 다릅니다. 매출 리스트를 통째로 접었는지 확인하세요'
assert check['합계'] == report['total_revenue'], '검산이 3단계의 total_revenue 와 다릅니다. 같은 매출을 두 길로 낸 것이니 같아야 합니다'
assert check['최고상품'] == '무선이어폰' and check['최저상품'] == '노트북거치대', 'head·last 가 다릅니다. collect 로 접기 전에 매출 내림차순으로 줄을 세웠는지 확인하세요'
assert [(r['순위'], r['상품']) for r in ranked] == [(1, '무선이어폰'), (2, '담요'), (3, '텀블러')], '순위표가 다릅니다. range 로 만든 자리 번호에 1 을 더했는지, 정렬이 같은지 확인하세요'
assert [r['상품'] for r in ranked] == top3, 'range 로 뽑은 상위 3 이 1단계의 top3 와 다릅니다. 두 길은 같은 답을 내야 합니다'
print('✅ 통과!')

---
수고했어요! LV3 에서 집계·랭킹·인덱스를 엮어 **추천 엔진**과 **판매 리포트**를 완성하고, 낸 숫자를 **리스트로 접어 검산**까지 했습니다. 다음 단원부터는 파일로 된 실제 데이터를 대량 적재해 **지식그래프를 구축**합니다.